In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

country = "Kenya"   # ← Change only this line for each country

df = pd.read_csv(f"{country.lower()}.csv")

df['Country'] = country

df['Date'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j', errors='coerce')
df['Month'] = df['Date'].dt.month
df['Year'] = df['Date'].dt.year
df.set_index('Date', inplace=True)

df = df.replace(-999, np.nan)

print("Duplicates:", df.duplicated().sum())
df = df.drop_duplicates()

print(df.describe())

missing_pct = (df.isna().sum() / len(df)) * 100
print("Missing %:\n", missing_pct[missing_pct > 0])

# Use only existing numeric weather columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
vars_for_z = [col for col in ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX'] if col in numeric_cols]

if vars_for_z:
    z_scores = np.abs(stats.zscore(df[vars_for_z], nan_policy='omit'))
    outliers = (z_scores > 3).any(axis=1)
    print("Outliers:", outliers.sum())

df[vars_for_z] = df[vars_for_z].ffill()
df = df.dropna(thresh=int(0.7 * df.shape[1]))

import os
os.makedirs('data', exist_ok=True)
df.to_csv(f'data/{country.lower()}_clean.csv', index=True)
print(f"Saved: data/{country.lower()}_clean.csv")